# ⚡ BoneTalk — Surface EMG Deep Learning on NVIDIA Tesla T4 GPU

This Google Colab notebook trains a **1D Temporal Convolutional Neural Network (1D-CNN)** on multi-channel surface EMG signals using an **NVIDIA Tesla T4 GPU** with **FP16 Mixed Precision (`torch.cuda.amp`)**.

### Key Principles:
- **Scientific Rigor**: Zero data leakage (session/recording-disjoint StratifiedGroupKFold splits), standard Cross-Entropy loss, legitimate scikit-learn evaluation.
- **No 3.9 GB Upload Needed**: Uses pre-packaged compact `.npz` archives (**1.57 MB** for MVP or **23.0 MB** for Days benchmark).
- **Hardware Acceleration**: Automatic Tensor Core FP16 execution on NVIDIA Tesla T4.
- **Export**: Saves `bonetalk_emg_model.pt`, `label_mapping.json`, confusion matrix, and training curves ready to download for the BoneTalk FastAPI backend.

## 1. Hardware Diagnostic & Environment Verification
Check that the Colab runtime is equipped with an NVIDIA Tesla T4 GPU.

In [ ]:
# Check NVIDIA GPU status
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Model:           {gpu_name}")
    print(f"Compute Capability:  {props.major}.{props.minor}")
    print(f"Total VRAM:          {props.total_memory / (1024**3):.2f} GB")
    print("✓ Tesla T4 GPU detected! Hardware acceleration & FP16 AMP are ACTIVE.")
else:
    print("⚠ CUDA is not available. Please go to Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

## 2. Dataset Selection & Upload
Upload either `bonetalk_mvp_data.npz` (1.57 MB, 4 isolated classes: REST, YES, NO, THANK YOU) or `bonetalk_days_data.npz` (23.0 MB, 8 classes: Mon-Sun + REST) using the Colab file upload panel on the left.

In [ ]:
import os
from pathlib import Path

# Select dataset to train on
# Options: "mvp" (4 classes, 1.57 MB) or "days" (8 classes, 23.0 MB)
TASK = "mvp"

dataset_filename = "bonetalk_mvp_data.npz" if TASK == "mvp" else "bonetalk_days_data.npz"
data_path = Path(dataset_filename)

if not data_path.exists():
    print(f"{dataset_filename} not found in current directory.")
    print("Please upload it via the Colab Files sidebar or Google Drive mount.")
    from google.colab import files
    print(f"Opening file uploader for {dataset_filename}...")
    uploaded = files.upload()
else:
    print(f"✓ Found dataset package: {dataset_filename} ({data_path.stat().st_size / (1024*1024):.2f} MB)")

## 3. Define the PyTorch 1D Temporal CNN Architecture
A hierarchical multi-scale 1D Convolutional Neural Network with Residual Connections, Batch Normalization, Dropout, and Global Average Pooling designed specifically for 8-channel surface EMG.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Conv1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, stride=1, pool_size=2, dropout=0.25):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.act1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.act2 = nn.LeakyReLU(0.1, inplace=True)
        self.pool = nn.MaxPool1d(kernel_size=pool_size) if pool_size > 1 else nn.Identity()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        total_downsample = stride * pool_size
        if in_channels != out_channels or total_downsample > 1:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
                nn.MaxPool1d(kernel_size=pool_size) if pool_size > 1 else nn.Identity(),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.act2(self.bn2(self.conv2(out)))
        out = self.pool(out)
        out = self.dropout(out + res)
        return out

class BoneTalk1DCNN(nn.Module):
    def __init__(self, in_channels=8, num_classes=4, base_filters=32, dropout=0.3):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_filters, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.block1 = Conv1DBlock(base_filters, base_filters, kernel_size=7, pool_size=2, dropout=dropout * 0.8)
        self.block2 = Conv1DBlock(base_filters, base_filters * 2, kernel_size=5, pool_size=2, dropout=dropout)
        self.block3 = Conv1DBlock(base_filters * 2, base_filters * 4, kernel_size=3, pool_size=2, dropout=dropout)
        self.block4 = Conv1DBlock(base_filters * 4, base_filters * 8, kernel_size=3, pool_size=2, dropout=dropout * 1.2)
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.gmp = nn.AdaptiveMaxPool1d(1)
        
        emb_dim = (base_filters * 8) * 2
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        h = self.stem(x)
        h = self.block1(h)
        h = self.block2(h)
        h = self.block3(h)
        h = self.block4(h)
        gap = self.gap(h).squeeze(-1)
        gmp = self.gmp(h).squeeze(-1)
        emb = torch.cat([gap, gmp], dim=1)
        return self.classifier(emb)

print("✓ Model architecture defined successfully.")

## 4. PyTorch Dataset, EMG Augmentation & DataLoaders
Implements EMG sensor noise jittering, amplitude scaling, time shifting, and channel masking for training stability.

In [ ]:
import random
import json
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

class EMGDataAugmentation:
    def __init__(self, noise_std=0.02, scale_range=(0.85, 1.15), max_shift_ratio=0.1, channel_dropout_prob=0.15):
        self.noise_std = noise_std
        self.scale_range = scale_range
        self.max_shift_ratio = max_shift_ratio
        self.channel_dropout_prob = channel_dropout_prob

    def __call__(self, x):
        if self.noise_std > 0 and random.random() < 0.7:
            x = x + torch.randn_like(x) * self.noise_std
        if self.scale_range and random.random() < 0.7:
            x = x * random.uniform(*self.scale_range)
        if self.max_shift_ratio > 0 and random.random() < 0.5:
            shift_max = int(x.shape[1] * self.max_shift_ratio)
            if shift_max > 0:
                shift = random.randint(-shift_max, shift_max)
                x = torch.roll(x, shifts=shift, dims=1)
        if self.channel_dropout_prob > 0 and random.random() < 0.3:
            drop_idx = random.randint(0, x.shape[0] - 1)
            x = x.clone()
            x[drop_idx, :] = 0.0
        return x

class BoneTalkEMGDataset(Dataset):
    def __init__(self, signals, labels, groups=None, is_train=False):
        self.signals = signals
        self.labels = np.asarray(labels, dtype=np.int64)
        self.groups = np.asarray(groups) if groups is not None else np.zeros(len(labels), dtype=object)
        self.is_train = is_train
        self.augmenter = EMGDataAugmentation() if is_train else None

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        sig = self.signals[idx]
        label = self.labels[idx]
        group = self.groups[idx]
        if sig.shape[0] > sig.shape[1] and sig.shape[1] == 8:
            sig = sig.T
        mean = np.mean(sig, axis=1, keepdims=True)
        std = np.std(sig, axis=1, keepdims=True) + 1e-6
        sig = (sig - mean) / std
        tensor_sig = torch.from_numpy(sig.astype(np.float32))
        if self.is_train and self.augmenter is not None:
            tensor_sig = self.augmenter(tensor_sig)
        return tensor_sig, torch.tensor(label, dtype=torch.long), str(group)

def collate_emg_batch(batch):
    tensors, labels, groups = zip(*batch)
    max_len = max(t.shape[1] for t in tensors)
    padded = [F.pad(t, (0, max_len - t.shape[1]), "constant", 0.0) for t in tensors]
    return torch.stack(padded, dim=0), torch.stack(labels, dim=0), list(groups)

# Load packaged arrays
data = np.load(data_path, allow_pickle=True)
signals = list(data["signals"])
labels = data["labels"]
groups = data["groups"]
class_names = [str(c) for c in list(data["class_names"])]
num_classes = len(class_names)

# Stratified Group K-Fold (disjoint sessions, zero leakage)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_val_idx, test_idx = next(sgkf.split(signals, labels, groups))

train_val_sigs = [signals[i] for i in train_val_idx]
train_val_lbls = labels[train_val_idx]
train_val_grps = groups[train_val_idx]
sgkf_val = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
sub_tr_idx, sub_val_idx = next(sgkf_val.split(train_val_sigs, train_val_lbls, train_val_grps))

real_train_idx = train_val_idx[sub_tr_idx]
real_val_idx = train_val_idx[sub_val_idx]

# DataLoaders
train_loader = DataLoader(
    BoneTalkEMGDataset([signals[i] for i in real_train_idx], labels[real_train_idx], groups[real_train_idx], is_train=True),
    batch_size=16, shuffle=True, collate_fn=collate_emg_batch, pin_memory=True
)
val_loader = DataLoader(
    BoneTalkEMGDataset([signals[i] for i in real_val_idx], labels[real_val_idx], groups[real_val_idx], is_train=False),
    batch_size=16, shuffle=False, collate_fn=collate_emg_batch, pin_memory=True
)
test_loader = DataLoader(
    BoneTalkEMGDataset([signals[i] for i in test_idx], labels[test_idx], groups[test_idx], is_train=False),
    batch_size=16, shuffle=False, collate_fn=collate_emg_batch, pin_memory=True
)

print(f"✓ Zero-leakage data partition verified:")
print(f"  Training recordings:   {len(real_train_idx)}")
print(f"  Validation recordings: {len(real_val_idx)}")
print(f"  Held-out Test:         {len(test_idx)}")
print(f"  Classes ({num_classes}):          {class_names}")

## 5. Train Model on NVIDIA Tesla T4 GPU (FP16 Mixed Precision)
Trains the 1D-CNN with AdamW, Cosine Annealing learning rate schedule, and FP16 mixed precision acceleration.

In [ ]:
import time
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def get_autocast(dev):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type=dev.type, enabled=(dev.type == "cuda"))
    return torch.cuda.amp.autocast(enabled=(dev.type == "cuda"))

# Class weights to balance training
weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=labels[real_train_idx])
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float, device=device))

model = BoneTalk1DCNN(in_channels=8, num_classes=num_classes, dropout=0.3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 70
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_f1 = -1.0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

start_t = time.time()
print(f"Starting training on {device} with mixed precision (AMP)={use_amp}...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss, tr_preds, tr_targets = 0.0, [], []
    for bx, by, _ in train_loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        optimizer.zero_grad()
        with get_autocast(device):
            logits = model(bx)
            loss = criterion(logits, by)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item() * len(by)
        tr_preds.extend(torch.argmax(logits, dim=1).detach().cpu().numpy())
        tr_targets.extend(by.cpu().numpy())
    
    scheduler.step()
    tr_acc = accuracy_score(tr_targets, tr_preds)
    
    # Validation
    model.eval()
    vl_loss, vl_preds, vl_targets = 0.0, [], []
    with torch.no_grad():
        for bx, by, _ in val_loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
            with get_autocast(device):
                logits = model(bx)
                loss = criterion(logits, by)
            vl_loss += loss.item() * len(by)
            vl_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            vl_targets.extend(by.cpu().numpy())
            
    vl_acc = accuracy_score(vl_targets, vl_preds)
    vl_f1 = f1_score(vl_targets, vl_preds, average="macro", zero_division=0)
    
    history["train_loss"].append(tr_loss / len(tr_targets))
    history["val_loss"].append(vl_loss / len(vl_targets))
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["val_f1"].append(vl_f1)
    
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_f1": float(vl_f1),
            "val_acc": float(vl_acc),
            "num_classes": int(num_classes),
            "class_names": class_names,
        }, "bonetalk_emg_model.pt")
        is_best = True
    else:
        is_best = False
        
    if epoch % 5 == 0 or epoch == 1 or is_best:
        print(f"Epoch [{epoch:2d}/{EPOCHS:2d}]  Train Loss: {tr_loss/len(tr_targets):.4f} | Val Loss: {vl_loss/len(vl_targets):.4f} | Val Acc: {vl_acc*100:5.2f}% | Val F1: {vl_f1:.4f} {'★ BEST' if is_best else ''}")

print(f"\n✓ Training finished in {time.time() - start_t:.2f} seconds!")

## 6. Held-Out Test Evaluation & Confusion Matrix
Evaluates the best saved checkpoint on the independent held-out test split.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

# Load best checkpoint
ckpt = torch.load("bonetalk_emg_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for bx, by, _ in test_loader:
        bx = bx.to(device, non_blocking=True)
        with get_autocast(device):
            logits = model(bx)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_targets.extend(by.numpy())

test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_prec = precision_score(test_targets, test_preds, average="macro", zero_division=0)
test_rec = recall_score(test_targets, test_preds, average="macro", zero_division=0)

print("=" * 65)
print("  BoneTalk 1D-CNN (Colab T4) — Held-out Test Results")
print("=" * 65)
print(f"  Test Accuracy:     {test_acc * 100:.2f}%")
print(f"  Macro F1 Score:    {test_f1:.4f}")
print(f"  Macro Precision:   {test_prec:.4f}")
print(f"  Macro Recall:      {test_rec:.4f}")
print(f"  Test Recordings:   {len(test_targets)}")
print(f"  Classes ({num_classes}): {class_names}")
print("=" * 65 + "\n")

print("Detailed Classification Report:")
print(classification_report(test_targets, test_preds, target_names=class_names, zero_division=0))

# Confusion Matrix Plot
cm = confusion_matrix(test_targets, test_preds, labels=np.arange(num_classes))
fig, ax = plt.subplots(figsize=(max(6, num_classes + 1), max(5, num_classes)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_ylabel("Actual Label")
ax.set_xlabel("Predicted Label")
ax.set_title(f"BoneTalk 1D-CNN Confusion Matrix (Test Acc: {test_acc*100:.1f}%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix_test.png", dpi=160)
plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"], label="Val Loss")
ax1.set_title("Loss Curve")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([a * 100 for a in history["train_acc"]], label="Train Acc %")
ax2.plot([a * 100 for a in history["val_acc"]], label="Val Acc %")
ax2.plot([f * 100 for f in history["val_f1"]], label="Val F1 %", linestyle="--")
ax2.set_title("Accuracy & F1 Curves")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=160)
plt.show()

## 7. Export Model Artifacts & Download
Saves `label_mapping.json`, exports the ONNX model, packages all artifacts into `bonetalk_colab_artifacts.zip`, and triggers download.

In [ ]:
import zipfile

# 1. Save label mapping
label_map = {str(i): name for i, name in enumerate(class_names)}
with open("label_mapping.json", "w") as f:
    json.dump({"id_to_label": label_map, "classes": class_names}, f, indent=2)

# 2. Export ONNX (optional)
try:
    dummy_input = torch.randn(1, 8, 800, device=device)
    torch.onnx.export(
        model, dummy_input, "bonetalk_emg_model.onnx",
        input_names=["emg_input"], output_names=["logits"],
        dynamic_axes={"emg_input": {0: "batch_size", 2: "time_steps"}, "logits": {0: "batch_size"}},
        opset_version=14
    )
    print("✓ Exported bonetalk_emg_model.onnx")
except Exception as e:
    print(f"ONNX export notice: {e}")

# 3. Zip all artifacts
files_to_zip = [
    "bonetalk_emg_model.pt",
    "label_mapping.json",
    "confusion_matrix_test.png",
    "training_curves.png"
]
if os.path.exists("bonetalk_emg_model.onnx"):
    files_to_zip.append("bonetalk_emg_model.onnx")

zip_filename = "bonetalk_colab_artifacts.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_zip:
        if os.path.exists(f):
            zipf.write(f)

print(f"✓ Successfully bundled artifacts into {zip_filename} ({os.path.getsize(zip_filename)/(1024*1024):.2f} MB)")

# Trigger browser download in Colab
try:
    from google.colab import files
    files.download(zip_filename)
    print("Downloading zip archive...")
except Exception:
    print(f"To download manually, locate '{zip_filename}' in the left sidebar Files panel.")